# 怎样亲手写出神经网络的训练过程？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

训练形状： torch.Size([160, 1])


可训练参数： 73


{
  "test_mse": 0.014536291360855103,
  "mean_baseline_mse": 0.9138361745687907,
  "train_first_loss": 2.533810615539551,
  "train_last_loss": 0.012649884447455406,
  "elapsed_seconds": 5.339
}


## 1. 补全：核对梯度与一步更新
对 (a−5)² 在 a=2 求导，按学习率 0.1 手动更新，断言新值为 2.6。

In [2]:
a = torch.tensor(2.0, requires_grad=True)
((a - 5) ** 2).backward()
with torch.no_grad():
    a -= 0.1 * a.grad
assert abs(a.item() - 2.6) < 1e-06
print(a.item())

2.5999999046325684


## 2. 修改：改为绝对误差
用当前预测计算 MAE，并与直接 NumPy 计算核对；解释它为何不像 MSE 一样放大大误差。

In [3]:
with torch.no_grad():
    mae = float(nn.L1Loss()(model(x_test), y_test))
assert np.isclose(mae, np.abs(predicted - y[160:, 0]).mean())
assert mae >= 0
print('测试 MAE', mae)

测试 MAE 0.0954066812992096


## 3. 迁移：学习线性计量关系
另生成 y=3x−2，无噪声。只用一个 Linear 层训练，最后检查斜率、截距；不要复用非线性答案。

In [4]:
torch.manual_seed(2)
xx = torch.linspace(-1, 1, 80).reshape(-1, 1)
yy = 3 * xx - 2
linear = nn.Linear(1, 1)
opt = torch.optim.SGD(linear.parameters(), lr=0.15)
for _ in range(150):
    opt.zero_grad()
    loss = ((linear(xx) - yy) ** 2).mean()
    loss.backward()
    opt.step()
assert abs(linear.weight.item() - 3) < 0.02 and abs(linear.bias.item() + 2) < 0.02
print(linear.state_dict())

OrderedDict({'weight': tensor([[3.0000]]), 'bias': tensor([-2.0000])})
